In [1]:
import pandas as pd
import json
from random import shuffle

# 数据说明

In [2]:
news = pd.read_csv('./data/MIND/news.tsv', sep='\t', header=None)

In [3]:
news.columns = ["ID", "类别", "子类别", "标题", "摘要", "链接", "标题实体", "摘要实体"]

In [4]:
news.head(3)

,ID,类别,子类别,标题,摘要,链接,标题实体,摘要实体
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."


In [5]:
news.shape

(51282, 8)

In [6]:
behaviors = pd.read_csv('./data/MIND/behaviors.tsv', sep='\t', header=None)
behaviors = behaviors.fillna('')

In [7]:
behaviors.columns = ["曝光ID", "用户ID", "曝光时间", "曝光前的新闻点击历史", "曝光明细 (1表示点击；0表示非点击)"]

In [8]:
behaviors.head(33)

,曝光ID,用户ID,曝光时间,曝光前的新闻点击历史,曝光明细 (1表示点击；0表示非点击)
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
3,4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
4,5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...
5,6,U19739,11/11/2019 6:52:13 PM,N39074 N14343 N32607 N32320 N22007 N442 N19001...,N21119-1 N53696-0 N33619-1 N25722-0 N2869-0
6,7,U8355,11/11/2019 12:22:09 PM,N8419 N15771 N1431 N5888 N18663 N24123 N22130 ...,N51346-0 N33848-0 N15132-0 N10688-0 N6342-0 N6...
7,8,U46596,11/12/2019 10:29:36 PM,N47438 N20950 N21317 N5469,N7821-0 N24898-0 N12029-0 N13579-0 N42977-0 N3...
8,9,U79199,11/13/2019 10:13:02 AM,N37083 N459 N29499 N38118 N37378 N24691 N27235...,N51048-1 N64094-0 N13907-0 N39010-0
9,10,U53231,11/11/2019 11:28:11 AM,N58936 N15919 N11917 N2153 N55312 N13008 N4142...,N53585-1 N55689-0


# 数据预处理

In [9]:
# 第一步：读取 news 获取 news ID 和 标题做映射 
new_ids = news["ID"].values.tolist()
news_titles = news["标题"].values.tolist()
news_dict = dict(zip(new_ids, news_titles))

In [10]:
# 总样本
samples = []
# 指令
instruction = "You are a news recommendation expert. Given the news click history (if the user has no news click history, indicate it as no click history) of the user and the news they like and dislike to watch, please decide whether the user likes to watch the target news by outputting \"Yes.\" or \"No.\""

In [11]:
# 第二步：读取 behaviors，获取用户喜欢和不喜欢的新闻
for idx, row in behaviors.iterrows():
    try: 
        sample_input = ""
        # print(f'曝光ID: {row["曝光ID"]}, 用户ID: {row["用户ID"]}')
        # 点击历史
        click_news_histories = []
        if len(row["曝光前的新闻点击历史"]) > 0:
            click_news = row["曝光前的新闻点击历史"].split(" ")
            
            # 如果大于3个，则直接选最后点击的3个
            if len(click_news) >= 3:
                click_news = click_news[-3:]
                
            for each_news in click_news:
                click_news_histories.append("\"" + news_dict[each_news] + "\"")
        else:
            click_news_histories.append('the user do not has click history.')
        # 曝光
        impressions = row["曝光明细 (1表示点击；0表示非点击)"].split(" ")
        likes = []
        dislikes = []
        # 前面的新闻做为训练数据，最后一个新闻做为预测
        for impression in impressions[:-1]:
            news_id, click = impression.split("-")
            news_title = news_dict[news_id]
            if int(click) == 1:
                likes.append("\""+news_title+"\"")
            else:
                dislikes.append("\""+news_title+"\"")

        sample_input = "User click histories: " + ", ".join(click_news_histories) + "\n"
        sample_input += "User likes: " + ', '.join(likes) + "\n" + "User dislikes: " + ', '.join(dislikes)

        news_id, click = impressions[-1].split("-")
        output = "Yes." if int(click) == 1 else "No."
        sample_input = sample_input + "\n" + "Whether the user will like the target news " + "\"" + news_dict[news_id] + "\"?"
        sample = {
            "instruction": instruction,
            "input": sample_input,
            "output": output,
            "用户ID": row["用户ID"]
        }
        samples.append(sample)
    except Exception as e:
        print(f'Parse {row["曝光ID"]} has exception: {e}')

In [12]:
# 查看一下样本
samples[25:30]

[{'instruction': 'You are a news recommendation expert. Given the news click history (if the user has no news click history, indicate it as no click history) of the user and the news they like and dislike to watch, please decide whether the user likes to watch the target news by outputting "Yes." or "No."',
  'input': 'User click histories: "Biggest private coal miner goes bust as Trump rescue fails", "McDonald\'s apologizes for \'Sundae Bloody Sundae\' Halloween promotion", "New York lawmakers are considering a ban on tackle football for kids under 12"\nUser likes: "Charles Rogers, former Michigan State football, Detroit Lions star, dead at 38"\nUser dislikes: "Porsche launches into second story of New Jersey building, killing 2", "Russell Wilson vs. 49ers\' No. 1 defense: Monday night\'s prime-time matchup", "60+ Restaurants Where Veterans Eat for Free on Veterans Day This Year", "Amazon\'s $1.5 million political gambit backfires in Seattle City Council election", "Andrew Yang\'s Cam

In [13]:
# 第三步 划分 train 和 test 保存样本
# 首先打乱
shuffle(samples)

train = samples[:int(len(samples)*0.8)]
test = samples[int(len(samples)*0.8):]

print(f'总样本数: {len(samples)}，训练集样本数: {len(train)}，测试集样本数: {len(test)}')

with open("./data/processed/mind_train.json", "w", encoding='utf-8') as save_file:
    json.dump(train, save_file, indent=4)
    
with open("./data/processed/mind_test.json", "w", encoding='utf-8') as save_file:
    json.dump(test, save_file, indent=4) # , sort_keys=True

总样本数: 156965，训练集样本数: 125572，测试集样本数: 31393
